# 20. Hierarchical Clustering
**Use case:** Financial Services — Portfolio diversification with correlation clustering and HRP

A portfolio that holds 25 stocks may still be concentrated if many of them move together, and that concentration shows up painfully in a market sell-off. Traditional mean-variance optimisation produces extreme, unstable weights because it inverts a noisy covariance matrix. Daily return history is available for every holding. Success means clusters that reveal what really moves together, a robust allocation that spreads risk across clusters, and lower out-of-sample volatility with low turnover than naive alternatives.

> 💡 It is like building a family tree for a football league: first pair up players who play almost identically, then merge those pairs into positions, then positions into defence and attack. You can stop at any level - 2 groups, 5 groups or 11 - by deciding where to cut the tree.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Merge 5 stocks step by step by correlation
Compute correlation distances between 5 stocks and perform agglomerative average-linkage merges by hand to see which assets belong together.

| 5W + H | |
|---|---|
| **Who** | A portfolio analyst at a wealth manager learning how diversification can be measured. |
| **What** | Group two banks, two tech stocks and a utility by how similarly their monthly returns move. |
| **When** | When reviewing a client portfolio that looks diversified by name count but may not be. |
| **Where** | A short numpy script with 8 months of returns typed in. |
| **Why** | Holding many stocks that move together gives little diversification; clustering reveals hidden concentration. |
| **How** | Build the correlation matrix, convert to distance sqrt(0.5 x (1 - corr)), start with each stock alone and repeatedly merge the two clusters with the smallest average distance. |

**🧠 ML concepts:** Agglomerative (bottom-up) clustering, Correlation as similarity, Correlation distance, Average linkage, Merge heights as dissimilarity  
**🐍 Python concepts:** Dict of lists, np.corrcoef, zip to pair names and rows, Tuples as cluster members, min() over a generator of tuples, enumerate with a filter

In [ ]:
# Easy: group 5 stocks by how alike their returns move - agglomerative merging by hand
import numpy as np

# 🐍 PY: dict of ticker -> list of 8 monthly returns in % (synthetic)
returns = {
    "BANK1": [2.1, -1.5, 3.0, 0.5, -2.2, 1.8, 0.9, -0.4],
    "BANK2": [1.9, -1.2, 2.7, 0.8, -2.5, 1.5, 1.1, -0.6],
    "TECH1": [4.5, 2.0, -3.1, 5.2, 1.0, -2.4, 3.8, 2.2],
    "TECH2": [4.0, 2.4, -2.8, 4.6, 1.5, -2.0, 3.1, 2.6],
    "UTIL1": [0.3, 0.6, 0.2, -0.1, 0.5, 0.4, -0.2, 0.1],
}
# 🐍 PY: list(dict) gives the keys in insertion order
names = list(returns)
R = np.array([returns[n] for n in names])

# 🧠 ML: correlation distance d = sqrt(0.5 * (1 - corr)): 0 = move together, 1 = move opposite
corr = np.corrcoef(R)
D = np.sqrt(0.5 * (1 - corr))
print("Correlation matrix:")
for n, row in zip(names, corr):
    # 🐍 PY: a generator inside join formats every number in the row
    print(f"  {n:<6}" + " ".join(f"{v:6.2f}" for v in row))

# 🧠 ML: start with every stock as its own cluster; repeatedly merge the two closest clusters
# 🐍 PY: each cluster is a tuple of member names so it can be used as a dict key / set element
clusters = [(n,) for n in names]

def cluster_dist(a, b):
    # 🧠 ML: AVERAGE linkage = mean distance between every member of a and every member of b
    # 🐍 PY: nested generator expression loops over both clusters' members
    return np.mean([D[names.index(x), names.index(y)] for x in a for y in b])

step = 1
while len(clusters) > 1:
    # 🐍 PY: build all (distance, i, j) pairs and min() picks the smallest distance first
    d, i, j = min((cluster_dist(clusters[i], clusters[j]), i, j)
                  for i in range(len(clusters)) for j in range(i + 1, len(clusters)))
    merged = clusters[i] + clusters[j]
    print(f"Step {step}: merge {clusters[i]} + {clusters[j]} at distance {d:.3f}")
    # 🐍 PY: keep every cluster except i and j, then add the new merged tuple
    clusters = [c for k, c in enumerate(clusters) if k not in (i, j)] + [merged]
    step += 1

# 🧠 ML: cutting the tree at a height (e.g. 0.5) gives the groups - one pick per group diversifies
print("\nReading: the two tech names merge first, then the banks; the utility joins last -> best diversifier.")

**Try this next:**
- Change cluster_dist to use min (single linkage) or max (complete linkage) and compare merge heights.
- Add a gold miner stock that moves opposite to tech and see where it merges.
- Make BANK2 noisier and see whether the banks still merge first.

## MEDIUM: scipy linkage on 20 stocks, cut into sector clusters
Cluster 20 stocks from 3 years of synthetic daily returns with scipy's Ward linkage, cut the tree into 4 groups and print a text dendrogram order.

| 5W + H | |
|---|---|
| **Who** | An investment risk team at an asset manager. |
| **What** | Discover sector-like groups among 20 stocks purely from how their returns co-move. |
| **When** | At each quarterly rebalance, or when adding new holdings. |
| **Where** | A Python analysis using scipy on daily return data from the market data platform. |
| **Why** | Official sector labels can mislead; return-based clusters show what actually moves together and so where risk is concentrated. |
| **How** | Simulate market, sector and stock-specific returns, compute correlation distance, run Ward linkage, cut with fcluster, read the dendrogram leaf order and compare clusters with true sectors. |

**🧠 ML concepts:** Linkage matrix Z, Ward linkage, Cutting a dendrogram into k clusters, Dendrogram leaf ordering, Within vs between cluster similarity  
**🐍 Python concepts:** Dict of sector to tickers, DataFrame column reordering by permutation, squareform conversion, groupby on a Series, Dict comprehension to invert a mapping, Boolean masks with broadcasting

In [ ]:
# Medium: cluster 20 stocks by return correlation with scipy linkage and cut into sector groups
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

np.random.seed(42)
N_DAYS = 750  # about 3 years of trading days

# 🐍 PY: dict of sector -> list of tickers (synthetic names)
sectors = {
    "Banks": ["BNK1", "BNK2", "BNK3", "BNK4", "BNK5"],
    "Tech": ["TEC1", "TEC2", "TEC3", "TEC4", "TEC5"],
    "Energy": ["ENG1", "ENG2", "ENG3", "ENG4", "ENG5"],
    "Utilities": ["UTL1", "UTL2", "UTL3", "UTL4", "UTL5"],
}
sector_vol = {"Banks": 0.012, "Tech": 0.016, "Energy": 0.018, "Utilities": 0.007}

# 🧠 ML: one-factor-per-sector model: return = market + sector factor + stock-specific noise
market = np.random.normal(0.0003, 0.008, N_DAYS)
cols = {}
for sector, tickers in sectors.items():
    factor = np.random.normal(0, sector_vol[sector], N_DAYS)
    for t in tickers:
        cols[t] = 0.6 * market + factor + np.random.normal(0, 0.008, N_DAYS)
# 🐍 PY: DataFrame from a dict of equal-length arrays -> one column per ticker
returns = pd.DataFrame(cols)
# 🐍 PY: shuffle the column order so the algorithm cannot "cheat" by position
returns = returns[np.random.permutation(returns.columns)]
print(f"Daily returns: {returns.shape[0]} days x {returns.shape[1]} stocks")
print(returns.iloc[:3, :6].round(4).to_string(), "\n")

# 🧠 ML: correlation -> distance. d = sqrt(0.5*(1-rho)) is a proper metric (0 = identical moves)
corr = returns.corr()
dist = np.sqrt(0.5 * (1 - corr))
# 🐍 PY: squareform converts the square matrix to the condensed 1-D form scipy expects
condensed = squareform(dist.values, checks=False)

# 🧠 ML: Ward merges the pair adding least within-cluster variance; valid here since this distance is Euclidean
# (sqrt(0.5*(1-rho)) is proportional to the straight-line gap between standardised return series)
Z = linkage(condensed, method="ward")
print("Last 5 merges (cluster_a, cluster_b, height, size):")
for row in Z[-5:]:
    print(f"  {int(row[0]):>3} + {int(row[1]):>3}  at height {row[2]:.3f}  -> {int(row[3])} stocks")

# 🧠 ML: cutting the tree into k groups - fcluster with criterion='maxclust'
labels = fcluster(Z, t=4, criterion="maxclust")
groups = pd.Series(labels, index=corr.index, name="cluster")
print("\nClusters found (cut at 4):")
# 🐍 PY: groupby on a Series then iterate (key, sub-series) pairs
for c, members in groups.groupby(groups):
    print(f"  Cluster {c}: {sorted(members.index)}")

# Text dendrogram: leaf order puts similar stocks next to each other
# 🐍 PY: no_plot=True returns the layout as a dict without drawing anything
leaf_order = dendrogram(Z, labels=list(corr.index), no_plot=True)["ivl"]
print("\nDendrogram leaf order:", " ".join(leaf_order))

# Check against the true sectors
# 🐍 PY: dict comprehension inverts sector->tickers into ticker->sector
true_sector = {t: s for s, ts in sectors.items() for t in ts}
check = pd.crosstab(groups, pd.Series(groups.index.map(true_sector), index=groups.index, name="true_sector"))
print("\nCluster vs true sector:\n" + check.to_string())

# Business interpretation: average correlation inside vs between clusters
same = groups.values[:, None] == groups.values[None, :]
off_diag = ~np.eye(len(groups), dtype=bool)
print(f"\nAvg correlation inside clusters : {corr.values[same & off_diag].mean():.2f}")
print(f"Avg correlation across clusters : {corr.values[~same].mean():.2f}")
print("Holding 5 stocks from ONE cluster is close to holding one stock - spread picks across clusters.")

**Try this next:**
- Switch method to 'single' or 'complete' and compare the last merge heights.
- Halve the sector volatilities and see if clusters still match sectors.
- Cut with criterion='distance' and t=0.8 instead of maxclust.

## HARD: Hierarchical Risk Parity vs equal weight, out of sample
Compare linkage methods by cophenetic correlation, build HRP-style weights from the cluster tree, and test volatility, concentration and turnover against equal-weight, inverse-variance and min-variance portfolios out of sample.

| 5W + H | |
|---|---|
| **Who** | A quantitative portfolio construction team at an asset manager or bank treasury. |
| **What** | A 25-stock allocation using Hierarchical Risk Parity, validated on a later, more volatile period. |
| **When** | At monthly or quarterly rebalances, with weights reviewed by the investment risk committee. |
| **Where** | A portfolio construction library whose weights feed the order management system. |
| **Why** | Classic minimum-variance optimisation inverts a noisy covariance matrix, giving extreme, unstable weights; HRP uses the cluster tree to spread risk more robustly. |
| **How** | Simulate returns with a stress regime, split train/test in time, compare single/average/complete/ward linkage, compute HRP weights by recursive bisection, evaluate test volatility, Sharpe, effective N and turnover, then save weights. |

**🧠 ML concepts:** Correlation distance as a metric, Cophenetic correlation to compare linkages, Quasi-diagonalisation via leaf order, Recursive bisection risk allocation (HRP), Out-of-sample evaluation across regimes, Weight stability, turnover and effective number of bets  
**🐍 Python concepts:** Helper functions with docstrings, np.r_ to join arrays, np.ix_ for sub-matrix selection, List comprehensions as a work queue, groupby with a function of the index, Dict of portfolios looped with items()

In [ ]:
# Hard: Hierarchical Risk Parity (HRP) - correlation clustering, linkage comparison, out-of-sample risk
import os
import tempfile
import joblib
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster, cophenet
from scipy.spatial.distance import squareform

rng = np.random.default_rng(42)
SECTORS = {"Banks": 0.012, "Tech": 0.017, "Energy": 0.020, "Health": 0.010, "Utilities": 0.006}
N_PER, N_DAYS = 5, 1000


def simulate_returns():
    """Sector-factor model with a mid-sample volatility regime change (stress period)."""
    market = rng.normal(0.0003, 0.009, N_DAYS)
    # 🐍 PY: np.r_ concatenates arrays - first half ones, second half 1.6
    stress = np.r_[np.ones(N_DAYS // 2), np.full(N_DAYS - N_DAYS // 2, 1.6)]
    cols = {}
    for sector, vol in SECTORS.items():
        factor = rng.normal(0, vol, N_DAYS)
        for i in range(1, N_PER + 1):
            # 🧠 ML: each stock has its own idiosyncratic risk level - unequal vols matter for allocation
            idio = rng.uniform(0.004, 0.015)
            cols[f"{sector[:3].upper()}{i}"] = stress * (0.7 * market + factor + rng.normal(0, idio, N_DAYS))
    df = pd.DataFrame(cols)
    # 🐍 PY: shuffle columns so clustering cannot rely on the original sector order
    return df[rng.permutation(df.columns)]


def corr_distance(returns):
    # 🧠 ML: d = sqrt(0.5 * (1 - rho)) turns correlation into a true distance metric
    return np.sqrt(0.5 * (1 - returns.corr()))


def inverse_variance_weights(cov):
    # 🧠 ML: inverse-variance = give each asset weight proportional to 1 / its variance
    iv = 1 / np.diag(cov)
    return iv / iv.sum()


def cluster_variance(cov, idx):
    # 🐍 PY: np.ix_ selects the sub-matrix for the given rows AND columns
    sub = cov[np.ix_(idx, idx)]
    w = inverse_variance_weights(sub)
    return float(w @ sub @ w)


def hrp_weights(returns, method="single"):
    """Lopez de Prado-style HRP: cluster, quasi-diagonalise, recursive bisection."""
    cov = returns.cov().values
    Z = linkage(squareform(corr_distance(returns).values, checks=False), method=method)
    # 🧠 ML: quasi-diagonalisation - order assets so correlated ones sit next to each other
    order = list(leaves_list(Z))
    # 🐍 PY: start every asset at weight 1.0, then multiply down as clusters are split
    w = pd.Series(1.0, index=order)
    # 🐍 PY: a list of lists acts as a work-queue of clusters still to split
    queue = [order]
    while queue:
        queue = [c[s:e] for c in queue for s, e in ((0, len(c) // 2), (len(c) // 2, len(c))) if len(c) > 1]
        # 🐍 PY: step through the queue two at a time - each pair is the left/right half of one split
        for i in range(0, len(queue), 2):
            left, right = queue[i], queue[i + 1]
            v_l, v_r = cluster_variance(cov, left), cluster_variance(cov, right)
            # 🧠 ML: give MORE money to the LESS risky half (risk-parity split)
            alpha = 1 - v_l / (v_l + v_r)
            w[left] *= alpha
            w[right] *= 1 - alpha
    return pd.Series(w.sort_index().values, index=returns.columns[w.sort_index().index])


def portfolio_stats(weights, returns):
    port = returns.values @ weights
    return port.std() * np.sqrt(252), (port.mean() * 252) / (port.std() * np.sqrt(252))


rets = simulate_returns()
# 🧠 ML: time split - estimate on the first half (calm), test on the second half (stress)
train, test = rets.iloc[: N_DAYS // 2], rets.iloc[N_DAYS // 2:]
print(f"{rets.shape[1]} stocks, train {len(train)} days, test {len(test)} days (higher-vol regime)\n")

# 1. Linkage comparison via cophenetic correlation and sector purity
# 🧠 ML: cophenetic corr = how faithfully the tree's merge heights preserve the original distances
cond = squareform(corr_distance(train).values, checks=False)
true_sector = [c[:3] for c in train.columns]
print("Linkage  | cophenetic corr | sector purity at 5 clusters")
for method in ["single", "average", "complete", "ward"]:
    Z = linkage(cond, method=method)
    coph, _ = cophenet(Z, cond)
    labels = fcluster(Z, 5, criterion="maxclust")
    # 🐍 PY: for each cluster, the share taken by its most common true sector, averaged by size
    purity = pd.crosstab(labels, true_sector).max(axis=1).sum() / len(labels)
    print(f"{method:<8} |      {coph:.3f}      |   {purity:.2f}")

# 2. Show the ordered clusters (quasi-diagonal order)
Z = linkage(cond, method="ward")
order = train.columns[leaves_list(Z)]
print("\nWard leaf order:", " ".join(order))

# 3. Build portfolios on TRAIN
# 🧠 ML: every portfolio is built ONLY from training data, then judged on the unseen test period
cov = train.cov().values
portfolios = {
    "Equal weight": np.full(train.shape[1], 1 / train.shape[1]),
    "Inverse variance": inverse_variance_weights(cov),
    "HRP (single)": hrp_weights(train, "single").values,
    "HRP (ward)": hrp_weights(train, "ward").values,
}
# Minimum-variance via inverse covariance - classic optimiser that HRP aims to stabilise
# 🧠 ML: inverting a noisy covariance matrix amplifies errors - the weakness HRP is designed to avoid
inv = np.linalg.inv(cov)
mv = inv @ np.ones(len(cov)) / (np.ones(len(cov)) @ inv @ np.ones(len(cov)))
portfolios["Min-variance (unconstrained)"] = mv

# 4. Evaluate in-sample and out-of-sample
print("\nPortfolio                    | train vol | test vol | test Sharpe | max weight | min weight")
for name, w in portfolios.items():
    # 🐍 PY: underscore _ ignores a return value we do not need
    tv, _ = portfolio_stats(w, train)
    ov, sr = portfolio_stats(w, test)
    print(f"{name:<28} |  {tv:6.2%}  | {ov:6.2%}  |   {sr:5.2f}     |  {w.max():6.2%}   | {w.min():7.2%}")

# 5. Explainability: weight by sector cluster
hrp = pd.Series(portfolios["HRP (ward)"], index=train.columns)
# 🐍 PY: groupby a function of the index (first 3 letters) aggregates weights per sector
print("\nHRP (ward) weight by sector:")
for sec, wt in hrp.groupby(lambda t: t[:3]).sum().sort_values(ascending=False).items():
    print(f"  {sec:<4} {wt:6.2%} " + "#" * int(wt * 100))

# 6. Stability: refit each method on the TEST window and measure how much the weights would move
# 🧠 ML: turnover = sum of |weight change|; unstable weights mean high trading cost and model risk
cov_te = test.cov().values
inv_te = np.linalg.inv(cov_te)
refit = {"HRP (ward)": hrp_weights(test, "ward").values,
         "Min-variance (unconstrained)": inv_te @ np.ones(len(cov_te)) / np.sum(inv_te)}
for name, w_new in refit.items():
    print(f"Rebalance turnover {name:<28}: {np.abs(w_new - portfolios[name]).sum():6.1%}")
same = np.allclose(portfolios["HRP (single)"], portfolios["HRP (ward)"])
print(f"HRP single vs ward identical? {same} (clean sector blocks give the same leaf order)")

# 7. Concentration check - effective number of bets
for name in ["Equal weight", "HRP (ward)", "Min-variance (unconstrained)"]:
    w = portfolios[name]
    # 🧠 ML: effective N = 1 / sum(w^2); 25 means perfectly spread, low values mean concentration
    print(f"Effective N ({name}): {1 / np.sum(w ** 2):5.1f}")

path = os.path.join(tempfile.mkdtemp(), "hrp_weights.joblib")
joblib.dump({"weights": hrp, "linkage": "ward", "as_of_day": N_DAYS // 2}, path)
print(f"\nSaved HRP weights -> {os.path.basename(path)}")
print("Model-risk note (SR 11-7): document distance, linkage and lookback choices; backtest across"
      " regimes; check weight stability before every rebalance.")

**Try this next:**
- Use 'average' linkage inside hrp_weights and compare test volatility.
- Shorten the training window to 120 days and watch min-variance turnover explode.
- Add a sixth sector with very high volatility and see how HRP down-weights it.